In [ ]:
import pandas as pd
import re
import unidecode
import os
import glob
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# pegando os arquivos mais recentes
def arquivo_mais_recente(padrao):
    arquivos = glob.glob(padrao)
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para o padrão: {padrao}")
    return max(arquivos, key=os.path.getctime)


In [ ]:
amazon_csv = arquivo_mais_recente("resultados_amazon/mais_vendidos_amazon_SAMPLE_*.csv")
ml_csv = arquivo_mais_recente("resultados/mais_vendidos_ml_SAMPLE_*.csv")

print("📄 Arquivo Amazon:", amazon_csv)
print("📄 Arquivo Mercado Livre:", ml_csv)

amazon = pd.read_csv(amazon_csv, sep=';', encoding='utf-8-sig')
mercadolivre = pd.read_csv(ml_csv, sep=';', encoding='utf-8-sig')


In [ ]:
col_nome_amz = "Nome"
col_nome_ml = "Nome"

col_cat_amz = "Subcategoria"
col_cat_ml = "Subcategoria"

col_nota = "Nota Geral"
col_qtd = "Qtd. Avaliações"



In [ ]:
# limpando os textos
def limpar_texto(texto):
    texto = str(texto).lower()
    texto = unidecode.unidecode(texto)
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    return texto.strip()


In [ ]:
def cortar_nome(texto, limite=100):
    return texto[:limite]


In [ ]:
amazon[col_nome_amz] = (
    amazon[col_nome_amz]
    .apply(limpar_texto)
    .apply(lambda x: cortar_nome(x, 100))
)

mercadolivre[col_nome_ml] = (
    mercadolivre[col_nome_ml]
    .apply(limpar_texto)
    .apply(lambda x: cortar_nome(x, 100))
)
# combinando categoria e nome para melhorar a similaridade

In [ ]:
### MAPA DE CATEGORIAS RELACIONADAS
MAPA_CATEGORIAS = {
    "Alimentos e Bebidas": ["Alimentos e Bebidas", "Esportes e Fitness", "Saúde"],
    "Automotivo": ["Acessórios para Veículos"],
    "Bebês": ["Bebês"],
    "Beleza": ["Beleza e Cuidado Pessoal"],
    "Beleza Premium": ["Beleza e Cuidado Pessoal"],
    "Brinquedos e Jogos": ["Brinquedos e Hobbies"],
    "Casa": ["Casa, Móveis e Decoração", "Agro"],
    "CD e Vinil": ["Música, Filmes e Seriados"],
    "Computadores e Informática": [
        "Informática", "Eletrônicos, Áudio e Vídeo", "Câmeras e Acessórios"
    ],
    "Cozinha": ["Eletrodomésticos", "Casa, Móveis e Decoração"],
    "Dispositivos Amazon e Acessórios": [
        "Informática", "Eletrônicos, Áudio e Vídeo"
    ],
    "DVD e Blu-ray": ["Música, Filmes e Seriados"],
    "Eletrodomésticos": ["Eletrodomésticos", "Casa, Móveis e Decoração"],
    "Eletrônicos": [
        "Eletrônicos, Áudio e Vídeo",
        "Celulares e Telefones",
        "Câmeras e Acessórios"
    ],
    "Esporte": ["Esportes e Fitness", "Saúde"],
    "Ferramentas e Mat. de Construção": ["Ferramentas", "Construção", "Agro"],
    "Games e Consoles": ["Games"],
    "Instrumentos Musicais": ["Instrumentos Musicais"],
    "Jardim e Piscina": ["Casa, Móveis e Decoração", "Agro"],
    "Livros": ["Livros, Revistas e Comics"],
    "Moda": ["Calçados, Roupas e Bolsas", "Joias e Relógios"],
    "Móveis": ["Casa, Móveis e Decoração"],
    "Papelaria e Escritório": ["Arte, Papelaria e Armarinho"],
    "Pet Shop": ["Pet Shop"],
    "Saúde e Bem-Estar": ["Saúde", "Beleza e Cuidado Pessoal", "Esportes e Fitness"]
}


In [ ]:
def media_ponderada(nota_amz, qtd_amz, nota_ml, qtd_ml):
    total = qtd_amz + qtd_ml

    if total == 0:
        return None

    return round(
        (nota_amz * qtd_amz + nota_ml * qtd_ml) / total,
        2
    )


In [ ]:
def converter_nota(valor):
    if pd.isna(valor):
        return 0.0
    return float(str(valor).replace(',', '.'))


In [ ]:
def converter_qtd(valor):
    if pd.isna(valor):
        return 0
    valor = re.sub(r'\D', '', str(valor))
    return int(valor) if valor else 0


In [ ]:
vectorizer = TfidfVectorizer()


In [ ]:
matches = []

for categoria_amz, categorias_ml in MAPA_CATEGORIAS.items():

    amazon_cat = amazon[amazon[col_cat_amz] == categoria_amz]
    ml_cat = mercadolivre[mercadolivre[col_cat_ml].isin(categorias_ml)]

    if amazon_cat.empty or ml_cat.empty:
        continue

    tfidf_amz = vectorizer.fit_transform(amazon_cat[col_nome_amz])
    tfidf_ml = vectorizer.transform(ml_cat[col_nome_ml])

    similarity_matrix = cosine_similarity(tfidf_amz, tfidf_ml)

    for i, (_, prod_amz) in enumerate(amazon_cat.iterrows()):
        idx_best = similarity_matrix[i].argmax()
        prod_ml = ml_cat.iloc[idx_best]

        # Amazon
        nota_amz = converter_nota(prod_amz[col_nota])
        qtd_amz = converter_qtd(prod_amz[col_qtd])

        # Mercado Livre
        nota_ml = converter_nota(prod_ml[col_nota])
        qtd_ml = converter_qtd(prod_ml[col_qtd])


        nota_final = media_ponderada(
            nota_amz, qtd_amz,
            nota_ml, qtd_ml
        )

        matches.append({
            "Categoria Amazon": categoria_amz,
            "Produto Amazon": prod_amz[col_nome_amz],
            "Nota Amazon": nota_amz,
            "Avaliações Amazon": qtd_amz,

            "Produto Mercado Livre": prod_ml[col_nome_ml],
            "Nota Mercado Livre": nota_ml,
            "Avaliações ML": qtd_ml,

            "Similaridade": round(similarity_matrix[i, idx_best], 3),
            "Nota Geral Ponderada": nota_final
        })


In [ ]:
resultados = pd.DataFrame(matches)


In [ ]:
limiar = 0.70
resultados_filtrados = resultados[resultados["Similaridade"] >= limiar]

print("🧾 Resultados filtrados:")
print(resultados_filtrados)


In [ ]:
saida_csv = f"comparacao_categorias_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.csv"

resultados_filtrados.to_csv(
    saida_csv,
    index=False,
    sep=';',
    encoding='utf-8-sig'
)

print(f"✅ Resultado salvo em: {saida_csv}")
